# Roman Disperser — Source Catalog & SED Formats
## Parquet Metadata Schema and Zarr SED Store

**Workshop Tutorial — Roman GRS PIT 2026**

The unified source catalog feeds the Roman grism simulation pipeline. It consists of **two** on-disk components:

| Component | Format | Contents |
|-----------|--------|----------|
| **`metadata.parquet`** | Apache Parquet | One row per source: sky position, morphology, F158 flux, SED lookup indices |
| **`seds.zarr/`** | Zarr v3 (sharded) | Wavelength grid + stellar SED templates + galaxy SEDs |

This notebook explores both formats, explains the schema, and shows how the disperser uses them to build grism images.

### Learning objectives

1. Understand the **Parquet metadata schema** — every column and its role in the pipeline
| Zarr SED store structure — wavelength grid, star SEDs, galaxy SED partitions
3. Learn how **`sed_index` + `flux_scale`** connect metadata to spectral data
4. See how the pipeline **loads, validates, and trims** SED data per-SCA
5. Build a **minimal stand-in catalog** for testing without the full Galacticus dataset

---

## 1. Why Two Separate Files?

The catalog is split into two files because each format is optimized for its data type:

| Format | Optimized For | Why Roman Disperser Uses It |
|--------|--------------|-----------------------------|
| **Parquet** | Tabular metadata | Columnar, filterable; cone searches skip loading SEDs entirely |
| **Zarr** | N-D numerical arrays | Sharding for random access; per-partition galaxy SEDs avoid loading 10+ GB at once |

**The separation enables**: loading the Parquet once (lightweight, ~50 MB for 400k sources), then loading **only the SEDs needed per SCA** from Zarr (typically ~200 MB vs ~7 GB for all cone galaxies).

**Directory structure:**

```
data/catalogs/
  metadata.parquet       # Source metadata (one row per source)
  seds.zarr/             # Zarr v3 store (directory)
    wavelengths          #   Common wavelength grid (float64)
    star_seds/           #   Stellar SED templates (N_templates × N_wl)
    galaxy_seds/         #   Group with per-partition arrays:
      sim_001/           #     Galaxy SEDs, partition 1
      sim_002/           #     Galaxy SEDs, partition 2
      ...
```

---

## 2. The Parquet Metadata Schema

The Parquet file is the **source of truth** for *which* sources to disperse and *where* on the detector they should appear. Each row is one source (star or galaxy).

### Required columns (consumed by the disperser)

These columns are read and used directly by `pipeline.py`:

| Column | Type | Units | Description |
|--------|------|-------|-------------|
| `ra` | float64 | deg | Right ascension (ICRS) |
| `dec` | float64 | deg | Declination (ICRS) |
| `type` | string | — | `"PSF"` (star/point source) or `"SER"` (Sérsic galaxy) |
| `n` | float32 | — | Sérsic index: 1.0 = exponential disk, 0 for stars |
| `half_light_radius` | float32 | arcsec | Half-light radius; 0 for stars |
| `pa` | float32 | deg | Position angle (E of N); 0 for stars |
| `ba` | float32 | — | Minor-to-major axis ratio (0–1); 1 for stars |
| `sed_index` | int32 | — | Row index into the Zarr SED array |
| `flux_scale` | float32 | — | SED multiplier (see Section 3) |
| `sim` | int16 | — | Partition number for galaxy SED lookup; 0 for stars |

### Ancillary columns (not used by the disperser)

These carry provenance and analysis metadata. The pipeline ignores them:

| Column | Type | Units | Description |
|--------|------|-------|-------------|
| `F158` | float32 | maggies | F158 apparent flux in linear AB units |
| `z_obs` | float32 | — | Observed redshift (includes peculiar velocity) |
| `z_cosmo` | float32 | — | Cosmological redshift |
| `src_index` | int32 | — | Row index in the original source file |

> **Key insight**: The Parquet file is **read once** at pipeline setup. Only the required columns are consumed by the disperser; ancillary columns are free space for analysis.

---

## 3. Inspecting the Metadata Parquet

Let's load the Parquet file and examine its structure.

In [ ]:
import pyarrow.parquet as pq
from pathlib import Path

# Resolve the catalog path (relative to this notebook)
catalog_dir = Path("../../../roman_disperser/data/catalogs")

# Read the metadata Parquet
meta = pq.read_table(catalog_dir / "metadata.parquet").to_pandas()

# Basic info
print(f"Total sources: {len(meta):,}")
print(f"Columns: {list(meta.columns)}")
print(f"\nSource type breakdown:")
print(meta["type"].value_counts())

### Shape of the data

The Galacticus 4 deg² catalog contains **400,000+ sources**, split between stars and galaxies. For a single pointing with a typical cone search radius (~0.6°), you'll select a few hundred to a few thousand sources.

Let's look at the column types and a sample of data:

In [ ]:
# Column types
print(meta.dtypes)

print("\n" + "="*80)

# First few rows as a sample
print("\nFirst 3 rows:")
display(meta.head(3))

print("\nDescriptive statistics for numeric columns:")
print(meta.describe())

---

## 4. Column Deep Dive

### `type`: PSF vs SER

The `type` column determines which disperser module is used:

- **`"PSF"`** — Point sources dispersed via [`star_disperser`](src/roman_disperser/star_disperser.py) (PSF deposition onto the detector)
- **`"SER"`** — Extended sources dispersed via [`galaxy_disperser`](src/roman_disperser/galaxy_disperser.py) (Sérsic morphology + Jacobian warping + PSF convolution)

### Morphology parameters (for `"SER"` sources only)

| Column | Meaning | Typical Range |
|--------|---------|---------------|
| `n` | Sérsic index: 0.3 (thin disk) to 4.0 (classical bulge) | 0.3–4.0 |
| `half_light_radius` | Angular half-light radius in arcseconds | 0.1"–5.0" |
| `pa` | Position angle, East of North | 0°–360° |
| `ba` | Minor-to-major axis ratio | 0.2–1.0 |

For `"PSF"` sources these columns are zeroed out (irrelevant — a star has no morphology).

### `sed_index` and `flux_scale`: linking Parquet to Zarr

| Column | What it does |
|--------|-------------|
| `sed_index` | Row index into the Zarr SED array. For stars: index into `star_seds`. For galaxies: row index within `galaxy_seds/sim_NNN` |
| `flux_scale` | SED amplitude multiplier. For stars: equals the F158 flux in **maggies**. For galaxies: always 1.0 (SEDs are already normalized to apparent flux) |

> **Maggie definition**: 1 maggie = 10^{-0.4 × mag} = the flux of a 0 mag AB source. The F158 column is also in maggies, so `flux_scale == F158` for stars.

### `sim`: Galaxy SED partitioning

The `sim` column assigns each galaxy to a partition (1–100 for the Galacticus catalog). Each partition is stored as a separate Zarr group (`galaxy_seds/sim_001`, `galaxy_seds/sim_002`, ...). This enables:

- Incremental catalog building (add a partition without rewriting others)
- Per-SCA loading (only load the partitions needed)
- Parallel building (one worker per partition)

---

## 5. How the Parquet Drives the Pipeline

The `build_grism_image.py` pipeline reads the Parquet file and performs these steps:

1. **Cone search** — filter sources near the pointing center (in degrees)
2. **Sky → FPA** — convert RA/Dec to focal plane coordinates using [`get_fpa_pos`](src/roman_disperser/optical_model_jax.py)
3. **FPA → SCA** — determine which sources fall on each detector using [`select_sources`](src/roman_disperser/catalog.py) (trace beam at 5 wavelengths, compute bounding box)
4. **SED lookup** — use `sed_index` to pull spectra from the Zarr store
5. **Flux scaling** — multiply SED by `flux_scale`
6. **Dispersion** — deposit or warp + convolve as appropriate

The Parquet file is **read once** at setup; the Zarr SEDs are loaded **per-SCA** to keep memory bounded.

Here's the cone search in action:

In [ ]:
# Simulate a cone search (like the pipeline does)
import numpy as np

# Pointing center (example: RA=9.5°, Dec=0.95° — a common test pointing)
ra0, dec0 = 9.5, 0.95
cone_radius_deg = 0.6  # degrees

# Approximate cone search (works for small radii near equator)
d_ra = (meta["ra"].values - ra0) * np.cos(np.deg2rad(dec0))
d_dec = meta["dec"].values - dec0
cone_mask = d_ra**2 + d_dec**2 < cone_radius_deg**2

n_cone = cone_mask.sum()
print(f"Sources within {cone_radius_deg}° of ({ra0}, {dec0}): {n_cone}")

cone_meta = meta[cone_mask]
print(f"\nType breakdown in cone:")
print(cone_meta["type"].value_counts())

print(f"\nSED index range for stars: [{cone_meta[cone_meta['type']=='PSF']['sed_index'].min()}, "
      f"{cone_meta[cone_meta['type']=='PSF']['sed_index'].max()}]")
print(f"SED index range for galaxies: [{cone_meta[cone_meta['type']=='SER']['sed_index'].min()}, "
      f"{cone_meta[cone_meta['type']=='SER']['sed_index'].max()}]")

---

## 6. The Zarr SED Store

### What is Zarr?

Zarr is an open-source format for chunked, compressed, N-dimensional arrays. The Roman disperser uses **Zarr v3** with:

- **Sharding** — each galaxy SED partition is a single file with an internal index, enabling efficient random access to non-consecutive sources
- **Compression** — blosc + zstd (level 3) with byte shuffle
- **Wavelength grid** — a single shared `wavelengths` array in Angstroms

### Zarr store structure

```
seds.zarr/
├── .zgroup                      # Zarr v3 group metadata
├── .zattrs                      # Global attributes
├── wavelengths                  # 1D array: [N_wavelength] float64
├── star_seds/                   # Group containing the star SED array
│   ├── .zgroup
│   └── .zarray                  # Array metadata (shape, dtype, compression)
└── galaxy_seds/                 # Group containing partitioned galaxy SEDs
    ├── .zgroup
    ├── sim_001/                 # Partition 1: [N_galaxies_in_part1, N_wavelength] float32
    ├── sim_002/                 # Partition 2
    └── ...
```

### Sharding: why it matters

Galaxy SED arrays use Zarr v3 sharding with:

| Parameter | Value | Purpose |
|-----------|-------|---------|
| Inner chunk | (10, N_wl) | Random access unit: reading 1 source decompresses only 10 rows (~220 KB compressed) |
| Shard (outer) | (N_sources, N_wl) | One shard file per partition on disk |

**Why this matters**: Without sharding (flat chunks of 1000 sources), gathering 1000 random sources takes ~21s. With sharding (inner chunks of 10), the same gather takes ~1.6s. This is critical for the per-SCA loading strategy.

---

## 7. Inspecting the Zarr Store

Let's open the Zarr store and examine its contents.

In [ ]:
import zarr
import numpy as np

# Open the Zarr store
store = zarr.open(str(catalog_dir / "seds.zarr"), mode="r")

print("Zarr store type:", type(store))
print("Top-level keys:", list(store.keys()))

# Wavelength grid
print("\n" + "="*80)
print("Wavelength grid:")
wavelengths = np.array(store["wavelengths"])
print(f"  Shape: {wavelengths.shape}")
print(f"  Min: {wavelengths.min():.0f} Å  ({wavelengths.min()/1e4:.2f} μm)")
print(f"  Max: {wavelengths.max():.0f} Å  ({wavelengths.max()/1e4:.2f} μm)")
print(f"  Spacing: {np.diff(wavelengths)[0]:.1f} Å")

# Star SEDs
print("\nStar SEDs:")
star_seds = store["star_seds"]
print(f"  Shape: {star_seds.shape}")
print(f"  Dtype: {star_seds.dtype}")

# Galaxy SEDs
print("\nGalaxy SEDs (partitions):")
for key in sorted(store["galaxy_seds"].keys()):
    arr = store["galaxy_seds"][key]
    print(f"  {key}: {arr.shape} ({arr.dtype})")

### Key observations

1. **Wavelength grid**: The catalog spans a broad range (typically 9000–21000 Å with 2 Å spacing). The grism only uses 9000–20000 Å, so the pipeline **trims** the grid to this range before use.

2. **Star SEDs**: Shape is `(N_templates, N_wavelength)`. Each row is a stellar atmosphere model (e.g., Pickles atlas). Templates are normalized to 0 AB magnitude in the F158 band (1 maggie).

3. **Galaxy SEDs**: Stored per-partition because the full galaxy SED array could be tens of gigabytes. Each partition is independent and can be loaded separately.

4. **Data types**: Star SEDs use `float32`, wavelengths use `float64`, galaxy SEDs use `float32`. The FLAM values are physical flux densities in **erg/s/cm²/Å**.

### Looking up a single SED

Here's the exact code pattern used by the pipeline to reconstruct a source spectrum from the catalog:

In [ ]:
# Look up one source (from the cone search results)
row = cone_meta.iloc[0]
print(f"Source {row.name}: type={row['type']}, sed_index={row['sed_index']}, "
      f"flux_scale={row['flux_scale']:.6e}, sim={row['sim']}")

if row["type"] == "PSF":
    sed = np.array(store["star_seds"][row["sed_index"]]) * row["flux_scale"]
    print(f"\nStar SED (template {row['sed_index']}):")
else:
    key = f"galaxy_seds/sim_{row['sim']:03d}"
    sed = np.array(store[key][row["sed_index"]]) * row["flux_scale"]
    print(f"\nGalaxy SED (partition {key}, row {row['sed_index']}):")

print(f"  Shape: {sed.shape}")
print(f"  Range: [{sed.min():.3e}, {sed.max():.3e}] erg/s/cm²/Å")
print(f"  Wavelength range: {wavelengths[0]:.0f}–{wavelengths[-1]:.0f} Å")

# Show the first 10 wavelength/flux pairs
print("\nFirst 10 wavelength/flux pairs:")
for i in range(10):
    print(f"  {wavelengths[i]:.0f} Å → {sed[i]:.3e} erg/s/cm²/Å")

---

## 8. SED Units and Wavelength Convention

### FLAM: Flux per unit wavelength

All SEDs store **f_λ (FLAM)** — apparent flux density per unit wavelength in units of **erg/s/cm²/Å**, stored as float32.

The count rate in a detector pixel is:

    counts/s = f_λ × sensitivity × Δλ

where `sensitivity` is the grism sensitivity curve and `Δλ` is the wavelength bin width in Angstroms.

### Star vs Galaxy SED normalization

| Source type | SED normalization | `flux_scale` |
|-------------|-------------------|-------------|
| **Stars** | Normalized to 0 AB mag in F158 (1 maggie) | `flux_scale = F158` (maggies) |
| **Galaxies** | Already in apparent FLAM | `flux_scale = 1.0` |

### Wavelength grid

| Property | Value |
|----------|-------|
| Unit | Angstroms (Å) |
| Range (catalog) | 9000–21000 Å |
| Spacing | 2 Å |
| Samples | ~6001 |
| Grism range (used) | 9000–20000 Å (0.9–2.0 μm) |

The wavelength grid is **shared** across all SEDs, ensuring consistent interpolation when computing dispersion for each source.

---

## 9. How SEDs Are Used in the Disperser

### For Stars: Simple Multiplication + PSF Deposition

The star disperser workflow:

1. **Lookup** SED by `sed_index` from `star_seds`
2. **Scale** by `flux_scale` (F158 magnitude → multiplier)
3. **Deposit** PSF onto detector with wavelength-dependent efficiency

```python
# From build_grism_image.py
sed = star_seds_all[sed_indices]      # [N_stars, N_wl] — FLAM spectra
sed_scaled = sed * flux_scales[:, None]  # [N_stars, N_wl]
output = star_disperser(sed_scaled, x_sca, y_sca, sensitivity)
```

### For Galaxies: Morphology + Spectrum + Jacobian Warping

The galaxy disperser workflow is more complex:

1. **Lookup** SED by `sim` + `sed_index` from `galaxy_seds/sim_NNN`
2. **Scale** by `flux_scale` (usually 1.0)
3. **Generate** a 2D Sérsic image from `n`, `half_light_radius`, `ba`, `pa`
4. **Compute Jacobian** of the dispersion mapping for each wavelength
5. **Warp** the Sérsic image via Jacobian (affine transform per wavelength)
6. **Convolve** with wavelength-dependent PSF
7. **Scale** by spectrum and sensitivity

```python
# From build_grism_image.py
sed = galaxy_seds[sim_partition][sed_index]   # [N_wl,] — FLAM
sed_scaled = sed * flux_scale
sersic_img = make_sersic_image(n, r_eff, ba, theta, npix_os)  # [npix, npix]
dispersed = galaxy_disperser(sed_scaled, sersic_img, x_sca, y_sca, sensitivity)
```

The key difference: **galaxies are spatially extended and warped** by the dispersion relation, while **stars are point sources** deposited at a single location (broadened by PSF).

---

## 10. From Catalog to Grism Image — The Full Data Flow

Here's a visual summary of how the catalog feeds into the disperser pipeline:

In [ ]:
from IPython.display import Image

# We'll draw the data flow as text for now
print("""
┌──────────────────────────────────────────────────────────────────┐
│                    metadata.parquet                              │
│  ra, dec, type, sed_index, flux_scale, n, ba, pa, sim, F158... │
└──────────────┬───────────────────────────────────────────────────┘
               │  cone search + sky→FPA→SCA
               ▼
┌──────────────────────────────────────────────────────────────────┐
│              Sources on detector (per SCA)                       │
│  ┌──────────┐  ┌──────────┐                                      │
│  │   Stars  │  │ Galaxies │                                      │
│  │ (PSF)    │  │  (SER)   │                                      │
│  └────┬─────┘  └────┬─────┘                                      │
│       │             │                                            │
│       ▼             │                                            │
│  ┌──────────┐  ┌──────────┐                                     │
│  │ star_seds│  │galaxy_seds│                                    │
│  │  [Zarr]  │  │  [Zarr]   │                                    │
│  └────┬─────┘  └────┬─────┘                                     │
│       │             │                                            │
│       ▼             │                                            │
│  flux × SED   morphology + flux × SED                           │
│       │             │                                            │
│       ▼             ▼                                            │
│  PSF deposition   Jacobian warp + PSF conv                       │
│       │             │                                            │
│       └─────┬───────┘                                            │
│             ▼                                                    │
│     Apply sensitivity curves                                     │
│             ▼                                                    │
│     Sum all orders (0, 1, 2)                                     │
│             ▼                                                    │
│         OUTPUT IMAGE                                             │
│    [DETECTOR_SIZE, DETECTOR_SIZE]                                │
└──────────────────────────────────────────────────────────────────┘
""")

---

## 11. Creating Stand-in Catalogs for Testing

If you don't have access to the full Galacticus catalog, you can create minimal stand-in catalogs to test the disperser pipeline. Here's how to build a tiny test catalog with a handful of sources:

> **Note**: This creates a synthetic catalog with placeholder SEDs. It's sufficient for testing the disperser logic but won't produce realistic spectra.

In [ ]:
import pyarrow as pa
import pyarrow.parquet as pq
import zarr
import numpy as np
from pathlib import Path

# Create a temporary test catalog
test_dir = Path("test_catalog")
test_dir.mkdir(exist_ok=True)

# ---- Build metadata.parquet ----
n_stars = 5
n_galaxies = 3

np.random.seed(42)
ra_star = np.random.uniform(9.4, 9.6, n_stars)
dec_star = np.random.uniform(0.9, 1.0, n_stars)
ra_gal = np.random.uniform(9.4, 9.6, n_galaxies)
dec_gal = np.random.uniform(0.9, 1.0, n_galaxies)

meta_rows = []

for i in range(n_stars):
    meta_rows.append({
        "ra": float(ra_star[i]),
        "dec": float(dec_star[i]),
        "type": "PSF",
        "n": 0.0,
        "half_light_radius": 0.0,
        "pa": 0.0,
        "ba": 1.0,
        "sed_index": i % 10,          # Reuse one of the first 10 star SEDs
        "flux_scale": float(1e-14 * (1 + 0.1 * i)),
        "sim": 0,
        "F158": float(15.0 + i * 0.5),
    })

for i in range(n_galaxies):
    meta_rows.append({
        "ra": float(ra_gal[i]),
        "dec": float(dec_gal[i]),
        "type": "SER",
        "n": float(1.0 + 0.5 * i),        # Exponential to bulge-like
        "half_light_radius": float(0.3 + 0.3 * i),  # 0.3" to 0.9"
        "pa": float(45.0 * i),
        "ba": float(0.7 - 0.1 * i),
        "sed_index": 0,                     # Reuse star SED 0 for simplicity
        "flux_scale": float(1e-14 * (1 + 0.1 * i)),
        "sim": 1,                            # Partition 1
        "F158": float(16.0 + i * 0.5),
    })

meta_df = pa.Table.from_pandas(
    __import__('pandas').DataFrame(meta_rows),
    preserve_index=False,
)
pq.write_table(meta_df, test_dir / "metadata.parquet")
print(f"Wrote {len(meta_rows)} sources to {test_dir / 'metadata.parquet'}")
print(f"  Stars: {n_stars}, Galaxies: {n_galaxies}")

---

## 12. Creating a Minimal Zarr SED Store

Now let's create the corresponding Zarr store with a wavelength grid and placeholder SEDs:

In [ ]:
# ---- Build seds.zarr ----
seds_dir = test_dir / "seds.zarr"
seds_dir.mkdir(exist_ok=True)

# Wavelength grid: 0.9–2.0 μm = 9000–20000 Å, 1 Å spacing
lam_min = 9000.0
lam_max = 20000.0
n_wl = int(lam_max - lam_min) + 1
wavelengths_test = np.arange(lam_min, lam_max + 1.0, 1.0, dtype=np.float64)

# Create Zarr store
store_root = zarr.open_group(str(seds_dir), mode="w", zarr_format=3)

# Wavelength grid
store_root.create_dataset(
    "wavelengths",
    data=wavelengths_test,
    chunks=wavelengths_test.shape,
    dtype="float64",
    fill_value=0.0,
    attributes={"description": "Wavelength grid in Angstroms"},
)

# Star SEDs: create 20 placeholder templates
n_star_templates = 20
star_seds = store_root.create_group("star_seds")
star_sed_data = np.zeros((n_star_templates, n_wl), dtype=np.float32)

for i in range(n_star_templates):
    wl_um = wavelengths_test / 1e4
    star_sed_data[i] = (
        wl_um**-4 * np.exp(-2.9e4 / wl_um) * 1e-14   # Rayleigh-Jeans tail
        + np.exp(-((wl_um - 1.2) / 0.05)**2) * 5e-15  # Narrow feature
        + np.exp(-((wl_um - 1.6) / 0.1)**2) * 2e-15   # Broad feature
    )

star_seds.create_dataset(
    "data",
    data=star_sed_data,
    chunks=(1, 100),              # Sharded: 1 SED per chunk, 100 wavelengths
    dtype="float32",
    fill_value=0.0,
    attributes={"description": "Stellar SED templates in FLAM (erg/s/cm²/Å)"},
)
print(f"Star SEDs: {star_sed_data.shape}")

# Galaxy SEDs: partition 1 with 5 templates
n_gal_templates = 5
gal_seds = store_root.create_group("galaxy_seds")
gal_partition = gal_seds.create_group("sim_001")
gal_sed_data = np.zeros((n_gal_templates, n_wl), dtype=np.float32)

for i in range(n_gal_templates):
    wl_um = wavelengths_test / 1e4
    gal_sed_data[i] = (
        wl_um**-1.5 * 1e-14         # Power law
        + np.exp(-((wl_um - 1.2) / 0.02)**2) * 1e-14  # [OIII]
        + np.exp(-((wl_um - 1.6) / 0.03)**2) * 5e-15   # Hα
    )

gal_partition.create_dataset(
    "data",
    data=gal_sed_data,
    chunks=(1, 100),
    dtype="float32",
    fill_value=0.0,
    attributes={"description": "Galaxy SED templates (partition 1)"},
)
print(f"Galaxy SEDs (sim_001): {gal_sed_data.shape}")

print(f"\nTest catalog created at: {test_dir}")
print(f"  metadata.parquet: {len(meta_rows)} sources")
print(f"  seds.zarr/: wavelengths + star_seds + galaxy_seds/sim_001")

---

## 13. Using the Test Catalog with the Disperser

With a test catalog in place, you can feed it directly into the disperser pipeline:

In [ ]:
# Verify the test catalog loads correctly
import zarr
import numpy as np

# Reload
test_meta = pq.read_table(test_dir / "metadata.parquet").to_pandas()
test_store = zarr.open(str(test_dir / "seds.zarr"), mode="r")

print("Metadata:")
print(test_meta[["ra", "dec", "type", "sed_index", "flux_scale"]])

print("\nWavelength grid:")
wl = np.array(test_store["wavelengths"])
print(f"  {len(wl)} samples from {wl.min():.0f} to {wl.max():.0f} Å")

print("\nStar SED shapes:", test_store["star_seds"]["data"].shape)
print("Galaxy SED shapes:", test_store["galaxy_seds"]["sim_001"]["data"].shape)

---

## 14. Validating Catalog Consistency

Before running the disperser, the pipeline validates catalog consistency. Here's a summary of the checks:

| Check | What it verifies | Error type |
|-------|-----------------|------------|
| Wavelength range | 9000–20000 Å covered | **Fatal** (ValueError) |
| Wavelength spacing | Uniform spacing | **Fatal** |
| Required columns | All 10 required columns present | **Fatal** |
| Type values | Only `"PSF"` and `"SER"` | **Fatal** |
| NaN/Inf in critical cols | `ra`, `dec`, `sed_index`, `flux_scale` | **Fatal** |
| sed_index bounds | Within star_seds array size | **Fatal** |
| Galaxy sim partitions | All referenced `sim` values exist | **Fatal** |
| Galaxy SED values | No pathological spikes > 1e-12 | **Warning** (zeroed) |

The **SED value limit** (1e-12) protects against catalog corruption: isolated single-bin spikes (e.g., sim_071/sed[555] = 3.5e28) would overflow float32 during FFT convolution and produce NaN/Inf output.

Here's how the pipeline scrubs bad SED bins:

In [ ]:
# SED scrubbing logic from the pipeline
def scrub_bad_sed_bins(sed_trimmed):
    """Zero out non-finite or out-of-range SED bins."""
    limit = 1e-12
    bad_mask = ~np.isfinite(sed_trimmed) | (np.abs(sed_trimmed) > limit)
    if bad_mask.any():
        n_bad = bad_mask.sum()
        print(f"  Scrubbing {n_bad} bad bins from SED")
        sed_trimmed = np.where(bad_mask, 0.0, sed_trimmed)
    return sed_trimmed

# Demonstrate with a pathological SED
bad_sed = np.ones(n_wl, dtype=np.float32) * 1e-15
bad_sed[555] = 3.5e28  # pathological spike (like sim_071/sed[555])
bad_sed[1200] = np.nan

print(f"Before scrubbing: max={bad_sed.max():.3e}, NaN count={np.isnan(bad_sed).sum()}")
clean_sed = scrub_bad_sed_bins(bad_sed.copy())
print(f"After scrubbing:  max={clean_sed.max():.3e}, NaN count={np.isnan(clean_sed).sum()}")

---

## 15. Workshop Discussion Questions

Click each question to reveal the answer.

<details>
<summary><b>Question 1:</b> Why separate stars (PSF) and galaxies (SER) into different disperser modules?</summary>

Stars are point sources — their dispersed image is just the PSF scaled by the spectrum at each wavelength. Galaxies are extended — you need to compute how the dispersion mapping *warps* the 2D morphology (via the Jacobian) before convolving with the PSF. The computational complexity is fundamentally different.

The star disperser deposits the PSF at a single (x, y) position for each wavelength. The galaxy disperser must:
1. Generate the Sérsic morphology image
2. Compute the Jacobian of the dispersion mapping (d(x_sca)/dλ, d(y_sca)/dλ)
3. Warp the image for each wavelength bin
4. Convolve with the wavelength-dependent PSF
5. Scale by the spectrum and sensitivity
</details>

<details>
<summary><b>Question 2:</b> Why store galaxy SEDs per-partition instead of one big array?</summary>

The full galaxy SED array can be tens of gigabytes. Partitioning allows:
- Incremental catalog building (new partition = new data, no rewrite of existing partitions)
- Per-SCA loading (only load the galaxy SEDs needed for each SCA)
- Memory management (avoid holding all galaxy SEDs in GPU memory)
- Parallel building (one worker per partition, no coordination needed)
</details>

<details>
<summary><b>Question 3:</b> What happens if a source's SED has a pathological spike (e.g., 3.5e28 at one wavelength)?</summary>

The pipeline scrubs any SED bin that exceeds 1e-12 (FLAM) or is non-finite. The bad bin is zeroed out, and a warning is printed. This prevents NaN/Inf from propagating into the output image during the FFT convolution in the galaxy disperser. The threshold of 1e-12 is far above any plausible physical SED but well below the smallest known pathological spike.
</details>

<details>
<summary><b>Question 4:</b> Why does the pipeline use FLAM instead of physical flux (erg/s/cm²/Å)?</summary>

FLAM (flux per unit wavelength) is the standard astronomical SED unit. It ensures consistent interpolation and scaling across different wavelength grids. The pipeline internally converts to photons/s when applying exposure time and throughput. The key advantage is that FLAM values are independent of wavelength spacing — the count rate is simply `f_λ × sensitivity × Δλ`.
</details>

<details>
<summary><b>Question 5:</b> How would you add a new source type (e.g., AGN with a power-law + broad-line SED) to this framework?</summary>

You'd need to:
1. Add new SED templates to `star_seds` (if point-like) or a new Zarr group (e.g., `agn_seds/` if extended)
2. Add a new `type` value (e.g., `"AGN"`) to the metadata Parquet schema
3. Update the disperser to route to the appropriate module (star_disperser for point AGN, galaxy_disperser for extended AGN)
4. Update the `validate_catalog` function to accept the new type
</details>

<details>
<summary><b>Question 6:</b> Why is the wavelength grid stored in Angstroms but the disperser uses microns?</summary>

The Zarr file stores wavelengths in Angstroms (standard astronomical practice), but the optical model and dispersion functions work internally in microns. The pipeline trims and converts in `build_grism_image.py`:

```python
wavelengths_ang, wl_mask, dlam = trim_wavelength_grid(wavelengths_full)
wavelengths_um = (wavelengths_ang / 1e4).astype(np.float32)
wavelengths_jax = jnp.array(wavelengths_um)
```

This conversion happens once at setup and is broadcast to all disperser functions.

---

## 16. Summary Cheat Sheet

### Metadata Parquet (`metadata.parquet`)

| Field | Type | Units | Used By |
|-------|------|-------|---------|
| `ra`, `dec` | float64 | deg | Cone search, sky→FPA |
| `type` | str | — | Star vs galaxy disperser |
| `n`, `hlr`, `pa`, `ba` | float32 | — | Galaxy Sérsic morphology |
| `sed_index` | int32 | — | Zarr SED lookup |
| `flux_scale` | float32 | — | SED amplitude scaling |
| `sim` | int16 | — | Galaxy SED partition |
| `F158` | float32 | maggies | Ancillary (not used by disperser) |

### Zarr Store (`seds.zarr/`)

| Array | Shape | Dtype | Notes |
|-------|-------|-------|-------|
| `wavelengths` | (N_wl,) | float64 | Å, 9000–21000 catalog, 9000–20000 trimmed |
| `star_seds` | (N_templates, N_wl) | float32 | FLAM, 0 AB mag F158 normalized |
| `galaxy_seds/sim_NNN` | (N, N_wl) | float32 | FLAM, apparent, sharded (10, N_wl) |

### Pipeline Data Flow

```
Parquet → cone search → sky→FPA→SCA
           │
           ├── PSF stars → star_seds → PSF deposition
           │
           └── SER galaxies → galaxy_seds → Sérsic → Jacobian warp → PSF conv
```

### Key Numbers

| Parameter | Value |
|-----------|-------|
| Catalog sources (4 deg²) | ~400,000 |
| Stars | Pickles atlas templates |
| Galaxies | Galacticus SEDs, 100 partitions |
| Wavelength range | 0.9–2.0 μm (5501 samples, 2 Å spacing) |
| Grism orders | 0 (2% efficiency), 1 (100%), 2 (1%) |
| Detector size | 4088 × 4088 pixels, 0.11"/pix |
| SED scrub limit | 1e-12 erg/s/cm²/Å |
| Galaxy SED loading | Per-SCA (~200 MB vs ~7 GB all-in) |
| Zarr compression | blosc+zstd, ~1.6× ratio |
| Zarr inner chunk | (10, N_wl) — 10 sources per chunk

---

## 17. References

- [Source Catalog README](../../../roman_disperser/data/catalogs/README.md) — Full format specification
- [`build_grism_image.py`](../../../roman_disperser/scripts/build_grism_image.py) — Pipeline entry point with catalog I/O
- [`catalog.py`](../../../roman_disperser/src/roman_disperser/catalog.py) — `select_sources()` for source assignment to detectors
- [`star_disperser.py`](../../../roman_disperser/src/roman_disperser/star_disperser.py) — Star (PSF) dispersion
- [`galaxy_disperser.py`](../../../roman_disperser/src/roman_disperser/galaxy_disperser.py) — Galaxy (Jacobian warp + PSF) dispersion
- [Zarr v3 Specification](https://zarr-specs.readthedocs.io/en/latest/v3/core/v3.html)
- [Apache Parquet Format](https://parquet.apache.org/docs/)
- [romanisim catalog convention](https://romanisim.readthedocs.io/en/latest/romanisim/catalog.html)

---

## Cleanup

Remove the test catalog directory when done:

In [ ]:
# Uncomment to clean up the test catalog:
# import shutil
# if test_dir.exists():
#     shutil.rmtree(test_dir)
#     print(f"Removed {test_dir}")